In [2]:
import pandas as pd

df = pd.read_excel("C:/Users/DELL/Downloads/hotel_daily_booking_data_2024_2025.xlsx",skiprows=1)
df['date'] = pd.to_datetime(df['date'])

In [3]:
df.head()

,date,room_type,bookings,price_inr,occupancy_pct,event_flag,event_name,day_of_week,is_weekend
0,2024-01-01,Deluxe,57,16000,89.7,1,New Year,Monday,0
1,2024-01-01,Suite,14,30650,64.5,1,New Year,Monday,0
2,2024-01-01,Standard,83,11350,86.8,1,New Year,Monday,0
3,2024-01-02,Deluxe,51,12950,81.6,0,NaN,Tuesday,0
4,2024-01-02,Suite,13,24150,61.5,0,NaN,Tuesday,0


In [4]:
room_counts = df.groupby(["room_type"])['room_type'].count()

In [5]:
room_counts

room_type
Deluxe      731
Standard    731
Suite       731
Name: room_type, dtype: int64

In [6]:
print(df.shape)

(2193, 9)


In [7]:
df = df.sort_values(['room_type','date'])

In [8]:
df['month'] = df['date'].dt.month
#df['day'] = df['date'].dt.day
df['week'] = df['date'].dt.isocalendar().week.astype(int)

In [9]:
df = df.sort_values(['room_type', 'date'])

df['lag_1'] = df.groupby('room_type')['bookings'].shift(1)


df['lag_7'] = df.groupby('room_type')['bookings'].shift(7)


df['rolling_7'] =  df.groupby('room_type')['bookings'].transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())

df['price_lag_1']      = df.groupby('room_type')['price_inr'].shift(1)
df['price_lag_7']      = df.groupby('room_type')['price_inr'].shift(7)
df['price_rolling_7']  = df.groupby('room_type')['price_inr'].transform(lambda x: x.shift(1).rolling(7).mean())


df = df.dropna(subset=['lag_1', 'lag_7', 'rolling_7', 'price_lag_1', 'price_lag_7', 'price_rolling_7'])


In [10]:
#df[(df['room_type'].isin(['Deluxe','Suite'])) & (df['date']<='2024-01-08')]
df[['lag_1', 'lag_7', 'rolling_7', 'price_lag_1', 'price_lag_7', 'price_rolling_7']].isna().sum()

lag_1              0
lag_7              0
rolling_7          0
price_lag_1        0
price_lag_7        0
price_rolling_7    0
dtype: int64

In [11]:
df.head()

,date,room_type,bookings,price_inr,occupancy_pct,event_flag,event_name,day_of_week,is_weekend,month,week,lag_1,lag_7,rolling_7,price_lag_1,price_lag_7,price_rolling_7
21,2024-01-08,Deluxe,34,8950,51.9,0,NaN,Monday,0,1,2,46.0,57.0,48.428571,10450.0,16000.0,11678.571429
24,2024-01-09,Deluxe,35,9400,53.8,0,NaN,Tuesday,0,1,2,34.0,51.0,45.142857,8950.0,12950.0,10671.428571
27,2024-01-10,Deluxe,32,9200,47.8,0,NaN,Wednesday,0,1,2,35.0,41.0,42.857143,9400.0,10700.0,10164.285714
30,2024-01-11,Deluxe,33,9050,51.4,0,NaN,Thursday,0,1,2,32.0,39.0,41.571429,9200.0,9450.0,9950.000000
33,2024-01-12,Deluxe,46,10400,71.5,0,NaN,Friday,1,1,2,33.0,47.0,40.714286,9050.0,10900.0,9892.857143


In [12]:
from sklearn.preprocessing import LabelEncoder
encoders = {}
for col in ['room_type','day_of_week']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

df = pd.get_dummies(df,columns=['event_name'],drop_first=True)

In [13]:
print(encoders['day_of_week'].classes_)
print(encoders['room_type'].classes_)
print(encoders)

['Friday' 'Monday' 'Saturday' 'Sunday' 'Thursday' 'Tuesday' 'Wednesday']
['Deluxe' 'Standard' 'Suite']
{'room_type': LabelEncoder(), 'day_of_week': LabelEncoder()}


In [14]:
for col, le in encoders.items():
    mapping = dict(zip(le.transform(le.classes_), le.classes_))
    print(f"{col}: {mapping}")


room_type: {np.int64(0): 'Deluxe', np.int64(1): 'Standard', np.int64(2): 'Suite'}
day_of_week: {np.int64(0): 'Friday', np.int64(1): 'Monday', np.int64(2): 'Saturday', np.int64(3): 'Sunday', np.int64(4): 'Thursday', np.int64(5): 'Tuesday', np.int64(6): 'Wednesday'}


In [15]:
rows = []
for col, le in encoders.items():
    for encoded, original in zip(le.transform(le.classes_), le.classes_):
        rows.append({'column': col, 'encoded': encoded, 'original': original})

pd.DataFrame(rows).to_excel("C:/Users/DELL/Downloads/label_encoding_mapping.xlsx", index=False)
#print("Saved ✅")


# ── Inverse transform later ──────────────────────────────────────────────────
print(encoders['room_type'].inverse_transform([0, 1, 2]))    # → ['Deluxe', 'Standard', 'Suite']
print(encoders['day_of_week'].inverse_transform([0, 1, 2,3,4,5,6]))  # → ['Friday', 'Monday', 'Saturday']


['Deluxe' 'Standard' 'Suite']
['Friday' 'Monday' 'Saturday' 'Sunday' 'Thursday' 'Tuesday' 'Wednesday']


In [16]:
#future['room_type'] = encoders['room_type'].inverse_transform(future['room_type'])
#encoders['room_type'].inverse_transform(future['room_type'])


In [17]:
print(df.columns)

Index(['date', 'room_type', 'bookings', 'price_inr', 'occupancy_pct',
       'event_flag', 'day_of_week', 'is_weekend', 'month', 'week', 'lag_1',
       'lag_7', 'rolling_7', 'price_lag_1', 'price_lag_7', 'price_rolling_7',
       'event_name_Diwali', 'event_name_New Year', 'event_name_Tamil New Year',
       'event_name_Valentine's Day'],
      dtype='object')


In [18]:
df = df.sort_values(['room_type', 'date'])

In [19]:
cutoff_date = (df['date'].max() - pd.Timedelta(days=30))
train_df = df[df['date'] <= cutoff_date]
val_df = df[df['date'] > cutoff_date]

print(cutoff_date)
print(train_df['date'].min(), train_df['date'].max())
print(val_df['date'].min(), val_df['date'].max())

2025-12-01 00:00:00
2024-01-08 00:00:00 2025-12-01 00:00:00
2025-12-02 00:00:00 2025-12-31 00:00:00


In [20]:
y = df['bookings']
features = [
    'room_type', 'day_of_week', 'is_weekend', 'month', 'week',
    'lag_1', 'lag_7', 'rolling_7',
    'price_lag_1', 'price_lag_7', 'price_rolling_7',   # ← new
    'event_name_Diwali', 'event_name_New Year',
    'event_name_Tamil New Year', "event_name_Valentine's Day"
]


In [21]:
X_train = train_df[features]
y_train = train_df['bookings']

X_val = val_df[features]
y_val = val_df['bookings']

In [22]:
X_train.dtypes

room_type                       int64
day_of_week                     int64
is_weekend                      int64
month                           int32
week                            int64
lag_1                         float64
lag_7                         float64
rolling_7                     float64
price_lag_1                   float64
price_lag_7                   float64
price_rolling_7               float64
event_name_Diwali                bool
event_name_New Year              bool
event_name_Tamil New Year        bool
event_name_Valentine's Day       bool
dtype: object

In [23]:
X_train.head()

,room_type,day_of_week,is_weekend,month,week,lag_1,lag_7,rolling_7,price_lag_1,price_lag_7,price_rolling_7,event_name_Diwali,event_name_New Year,event_name_Tamil New Year,event_name_Valentine's Day
21,0,1,0,1,2,46.0,57.0,48.428571,10450.0,16000.0,11678.571429,False,False,False,False
24,0,5,0,1,2,34.0,51.0,45.142857,8950.0,12950.0,10671.428571,False,False,False,False
27,0,6,0,1,2,35.0,41.0,42.857143,9400.0,10700.0,10164.285714,False,False,False,False
30,0,4,0,1,2,32.0,39.0,41.571429,9200.0,9450.0,9950.000000,False,False,False,False
33,0,0,1,1,2,33.0,47.0,40.714286,9050.0,10900.0,9892.857143,False,False,False,False


In [24]:
def evaluate(name, model, X, y):
    preds = model.predict(X)
    mae  = mean_absolute_error(y, preds)
    rmse = np.sqrt(mean_squared_error(y, preds))
    mape = np.mean(np.abs((y - preds) / y.clip(lower=1))) * 100
    print(f"{name:10s} | MAE: {mae:.2f} | RMSE: {rmse:.2f} | MAPE: {mape:.1f}%")
    return preds

In [25]:
import optuna
import numpy as np
import lightgbm as lgb
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

optuna.logging.set_verbosity(optuna.logging.WARNING)  # suppress trial logs

# ── 1. Tune XGBoost ─────────────────────────────────────────────────────────
def xgb_objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 200, 1000),
        'max_depth':         trial.suggest_int('max_depth', 3, 8),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
        'gamma':             trial.suggest_float('gamma', 0, 0.5),
        'early_stopping_rounds': 50,
        'eval_metric': 'mae',
        'random_state': 42
    }
    model = XGBRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return mean_absolute_error(y_val, model.predict(X_val))

xgb_study = optuna.create_study(direction='minimize')
xgb_study.optimize(xgb_objective, n_trials=50)
print("Best XGBoost MAE:", xgb_study.best_value)
print("Best XGBoost params:", xgb_study.best_params)


# ── 2. Tune LightGBM ────────────────────────────────────────────────────────
def lgbm_objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 200, 1000),
        'max_depth':         trial.suggest_int('max_depth', 3, 8),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'num_leaves':        trial.suggest_int('num_leaves', 20, 100),
        'random_state': 42,
        'verbose': -1
    }
    model = LGBMRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)])
    return mean_absolute_error(y_val, model.predict(X_val))

lgbm_study = optuna.create_study(direction='minimize')
lgbm_study.optimize(lgbm_objective, n_trials=50)
print("Best LightGBM MAE:", lgbm_study.best_value)
print("Best LightGBM params:", lgbm_study.best_params)


# ── 3. Train final models with best params ──────────────────────────────────
final_xgb = XGBRegressor(
    **xgb_study.best_params,
    early_stopping_rounds=50,
    eval_metric='mae',
    random_state=42
)
final_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

final_lgbm = LGBMRegressor(**lgbm_study.best_params)
final_lgbm.fit(X_train, y_train, eval_set=[(X_val, y_val)])

# ── 4. Compare ──────────────────────────────────────────────────────────────
#evaluate("XGBoost (tuned)",  final_xgb,  X_val, y_val)
#evaluate("LightGBM (tuned)", final_lgbm, X_val, y_val)

Best XGBoost MAE: 2.6196398735046387
Best XGBoost params: {'n_estimators': 691, 'max_depth': 8, 'learning_rate': 0.029422125232323657, 'subsample': 0.653859197870107, 'colsample_bytree': 0.9073003306327977, 'min_child_weight': 3, 'gamma': 0.03366224132370128}
Best LightGBM MAE: 2.873448285757389
Best LightGBM params: {'n_estimators': 454, 'max_depth': 5, 'learning_rate': 0.09413566287600035, 'subsample': 0.7029079426738268, 'colsample_bytree': 0.9110816653013657, 'min_child_samples': 17, 'num_leaves': 93}


,boosting_type,'gbdt'
,num_leaves,93
,max_depth,5
,learning_rate,0.09413566287600035
,n_estimators,454
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,17


In [26]:
xgb_best_pred = evaluate("XGBoost (Tuned) Test", final_xgb, X_val, y_val)
xgb_best_trainpred = evaluate("XGBoost (Tuned) Train", final_xgb, X_train, y_train)

XGBoost (Tuned) Test | MAE: 2.62 | RMSE: 3.58 | MAPE: 6.7%
XGBoost (Tuned) Train | MAE: 0.73 | RMSE: 0.98 | MAPE: 3.2%


In [27]:
lgbm_best_pred = evaluate("LightGBM (Tuned) Test", final_lgbm, X_val, y_val)
lgbm_best_trainpred = evaluate("LightGBM (Tuned) Train", final_lgbm, X_train, y_train)

LightGBM (Tuned) Test | MAE: 2.95 | RMSE: 3.91 | MAPE: 8.4%
LightGBM (Tuned) Train | MAE: 1.01 | RMSE: 1.41 | MAPE: 4.0%


In [32]:
val_result = val_df[['date','room_type','bookings']].copy()

val_result['prediction_xgb'] = xgb_best_pred
val_result['prediction_lgbm'] = lgbm_best_pred

val_result['median_bookings'] = val_result[['prediction_xgb','prediction_lgbm']].median(axis=1)
#val_result['type'] = 'Validation Prediction'

In [29]:
#val_result[val_result['room_type']==2]

In [33]:
val_result.head(10)

,date,room_type,bookings,prediction_xgb,prediction_lgbm,median_bookings
2103,2025-12-02,0,37,35.350208,37.415060,36.382634
2106,2025-12-03,0,32,37.029686,40.777835,38.903761
2109,2025-12-04,0,43,34.996105,32.821140,33.908623
2112,2025-12-05,0,52,54.819729,57.183449,56.001589
2115,2025-12-06,0,54,54.668560,57.045429,55.856995
2118,2025-12-07,0,46,48.154354,48.586624,48.370489
2121,2025-12-08,0,33,33.219730,32.731311,32.975521
2124,2025-12-09,0,39,34.661690,33.491851,34.076770
2127,2025-12-10,0,37,39.374519,38.648075,39.011297
2130,2025-12-11,0,39,39.286331,38.766140,39.026236


In [35]:
mae  = mean_absolute_error(val_result['bookings'], val_result['median_bookings'])
rmse = np.sqrt(mean_squared_error(val_result['bookings'], val_result['median_bookings']))
mape = np.mean(np.abs((val_result['bookings']-val_result['median_bookings']) / y.clip(lower=1))) * 100
print(f" MAE: {mae:.2f} | RMSE: {rmse:.2f} | MAPE: {mape:.1f}%")

 MAE: 2.74 | RMSE: 3.66 | MAPE: 7.3%
